# Prueba de funcionamiento del pipeline — Parte II

Este notebook **demuestra que el proyecto funciona de punta a punta**: ejecuta el pipeline completo desde los datos
crudos y comprueba, con `assert`, que el resultado es correcto, reproducible y trazable. Si una comprobación falla,
el notebook se detiene en ella con un mensaje que dice qué falló.

| # | Prueba | Qué demuestra |
| --- | --- | --- |
| 1 | Pipeline completo | `python pipeline.py` ejecuta las seis etapas sin errores |
| 2 | Reproducibilidad | Una nueva ejecución produce exactamente los mismos CSV finales (SHA-256) |
| 3 | Balance y trazabilidad | En cada etapa, filas de entrada = salida + excluidas, y cada exclusión tiene su registro |
| 4 | Invariantes del dataset final | Identificador único, rangos válidos y gamas completas en `homeoffice_final.csv` |
| 5 | Almacén SQLite | Las cinco tablas se consultan con SQL y la clave primaria rechaza duplicados |
| 6 | Detección de errores | Con un dato corrupto, el pipeline se detiene en la etapa correcta y nombra el problema |

Esta copia ya está **ejecutada**: los resultados de cada prueba se ven debajo de su celda, sin necesidad de correrla.

> **Para ejecutarlo usted mismo**: el notebook no funciona como archivo suelto, porque ejecuta el proyecto completo
> (`pipeline.py`, los notebooks 01–05 y los datos). Su ubicación en el repositorio es
> [`notebooks/prueba_de_funcionamiento.ipynb`](https://github.com/josueroc/proyectoacd_mercadolibre/blob/main/notebooks/prueba_de_funcionamiento.ipynb).
>
> 1. Descargar el repositorio (https://github.com/josueroc/proyectoacd_mercadolibre) con `git clone` o como ZIP, y descomprimirlo.
> 2. Instalar las dependencias: `pip install -r requirements.txt`.
> 3. Abrir `notebooks/prueba_de_funcionamiento.ipynb` (en Jupyter o VS Code) y ejecutar todas las celdas (*Run All*).
>
> Tarda uno o dos minutos. La prueba 6 trabaja sobre una copia temporal, así que no modifica los datos del repositorio.

In [1]:
import hashlib
import json
import os
import platform
import shutil
import sqlite3
import subprocess
import sys
import tempfile
import time
from contextlib import closing
from pathlib import Path

import pandas as pd
from IPython.display import display

# Raíz del repositorio: la carpeta que contiene pipeline.py (funciona desde notebooks/ o desde la raíz).
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pipeline.py').exists()), None)
if RAIZ is None:
    raise FileNotFoundError(
        'Este notebook debe ejecutarse dentro del repositorio completo, porque ejecuta todo el pipeline '
        '(pipeline.py, los notebooks 01–05 y los datos). Descargue el repositorio desde '
        'https://github.com/josueroc/proyectoacd_mercadolibre y abra notebooks/prueba_de_funcionamiento.ipynb.')
FINAL = RAIZ / 'data' / 'final'
EVIDENCIA = RAIZ / 'data' / 'evidencia'
resultados = []


def comprobar(prueba, demuestra, condicion, detalle):
    """Detiene el notebook si la condición no se cumple; si se cumple, la registra en el resumen final."""
    assert condicion, f'{prueba} FALLÓ: {detalle}'
    resultados.append({'prueba': prueba, 'qué demuestra': demuestra, 'resultado': f'OK · {detalle}'})
    print(f'OK  {prueba}: {detalle}')


def ejecutar_pipeline(*argumentos, carpeta=RAIZ):
    """Ejecuta pipeline.py como lo haría cualquier usuario desde la terminal y devuelve su salida y duración."""
    inicio = time.perf_counter()
    proceso = subprocess.run([sys.executable, 'pipeline.py', *argumentos], cwd=carpeta, capture_output=True,
                             text=True, encoding='utf-8', errors='replace',
                             env={**os.environ, 'PYTHONIOENCODING': 'utf-8'})
    return proceso, time.perf_counter() - inicio


def miles(n):
    return f'{n:,}'.replace(',', ' ')


def sha256(ruta):
    # Igual que el manifiesto: se normalizan los saltos de línea para que Windows y Linux den el mismo hash.
    return hashlib.sha256(ruta.read_bytes().replace(b'\r\n', b'\n')).hexdigest()


try:
    commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=RAIZ, capture_output=True,
                            text=True).stdout.strip() or 'sin git'
except FileNotFoundError:
    commit = 'sin git'
print(f'Repositorio: {RAIZ.name} · commit {commit}')
print(f'Python {platform.python_version()} · pandas {pd.__version__} · {platform.system()} {platform.release()}')

Repositorio: proyectoacd_mercadolibre · commit 1e478a5
Python 3.11.4 · pandas 2.1.0 · Windows 10


## Prueba 1 — El pipeline completo se ejecuta sin errores

Antes de ejecutar se guarda el manifiesto vigente (el versionado en GitHub, en una copia recién clonada), para
compararlo después en la prueba 2. Luego se ejecuta `python pipeline.py`: ingestión y limpieza → faltantes y
atípicos → integración y reducción → transformación e ingeniería de características → almacenamiento en SQLite →
análisis.

In [2]:
manifiesto_antes = json.loads((FINAL / 'manifiesto.json').read_text(encoding='utf-8'))

proceso, segundos = ejecutar_pipeline()
print(proceso.stdout)
if proceso.returncode != 0:
    print(proceso.stderr[-3000:])

etapas_ok = proceso.stdout.count('] OK en')
comprobar('1. Pipeline completo', 'El pipeline es funcional de punta a punta',
          proceso.returncode == 0 and etapas_ok == 6,
          f'{etapas_ok} de 6 etapas OK en {segundos:.0f} s, código de salida {proceso.returncode}')

[01] Ingestión y limpieza ...
[01] OK en 7.9 s
[02] Valores faltantes y atípicos ...
[02] OK en 13.0 s
[03] Integración y reducción ...
[03] OK en 7.1 s
[04] Transformación e ingeniería de características ...
[04] OK en 9.4 s
[05] Almacenamiento en SQLite ...
[05] OK en 0.4 s {'publicaciones': 2714, 'precios_por_gama': 18, 'perfiles': 3, 'ofertas_comunes': 767, 'excluidos': 1079}
[06] Análisis y visualización ...
[06] OK en 10.4 s
Pipeline completo en 48.2 s.

OK  1. Pipeline completo: 6 de 6 etapas OK en 51 s, código de salida 0


## Prueba 2 — La ejecución es reproducible

El manifiesto guarda el número de filas, de columnas y el SHA-256 (una "huella digital") de cada CSV final. Si la
huella de antes y la de después coinciden, los archivos son idénticos byte a byte. Además se recalcula cada huella
directamente sobre los archivos, sin confiar en el manifiesto.

In [3]:
manifiesto_despues = json.loads((FINAL / 'manifiesto.json').read_text(encoding='utf-8'))
tabla = pd.DataFrame([
    {'archivo': archivo, 'filas': datos['filas'], 'columnas': datos['columnas'],
     'SHA-256 antes': manifiesto_antes.get(archivo, {}).get('sha256', '—')[:16],
     'SHA-256 después': datos['sha256'][:16],
     'recalculado coincide': sha256(FINAL / archivo) == datos['sha256'],
     'idéntico': manifiesto_antes.get(archivo) == datos}
    for archivo, datos in manifiesto_despues.items()])
display(tabla)

comprobar('2. Reproducibilidad', 'Dos ejecuciones producen exactamente los mismos datos',
          tabla['idéntico'].all() and tabla['recalculado coincide'].all(),
          f'{len(tabla)} CSV finales con el mismo SHA-256 antes y después')

,archivo,filas,columnas,SHA-256 antes,SHA-256 después,recalculado coincide,idéntico
0,homeoffice_final.csv,2714,41,ac4599518a53ed3a,ac4599518a53ed3a,True,True
1,integracion_ofertas_comunes.csv,767,5,a7466782a35f77d5,a7466782a35f77d5,True,True
2,perfiles_home_office.csv,3,10,f06aedb9fe3969b9,f06aedb9fe3969b9,True,True
3,precios_por_gama.csv,18,8,d5553835359ee427,d5553835359ee427,True,True


OK  2. Reproducibilidad: 4 CSV finales con el mismo SHA-256 antes y después


## Prueba 3 — Ningún registro se pierde sin explicación

Cada notebook registra cuántas filas entran, cuántas salen y cuántas se excluyen por cada motivo. La prueba
comprueba que en todas las etapas *entrada = salida + excluidas* y que cada exclusión del home office tiene su fila en
`data/evidencia/`, con el motivo.

In [4]:
balance = json.loads((FINAL / 'metricas.json').read_text(encoding='utf-8'))['balance']
filas = []
for etapa, b in balance.items():
    excluidas = sum(v for k, v in b.items() if k not in ('entrada', 'salida'))
    motivos = ', '.join(f'{k} {v}' for k, v in b.items() if k not in ('entrada', 'salida')) or '—'
    filas.append({'etapa': etapa, 'entrada': b['entrada'], 'salida': b['salida'], 'excluidas': excluidas,
                  'motivos': motivos, 'cuadra': b['entrada'] == b['salida'] + excluidas})
balance_tabla = pd.DataFrame(filas)
display(balance_tabla)

# Cada exclusión del home office tiene su registro en data/evidencia/
evidencia = {'duplicados': '01_duplicados.csv', 'ajenas': '01_ajenas_al_componente.csv',
             'errores': '02_excluidos_por_error.csv'}
contados = {'duplicados': balance['01_homeoffice']['duplicados'], 'ajenas': balance['01_homeoffice']['ajenas'],
            'errores': balance['02_homeoffice']['errores']}
registrados = {motivo: len(pd.read_csv(EVIDENCIA / archivo)) for motivo, archivo in evidencia.items()}
print('Exclusiones contadas:   ', contados)
print('Registros en evidencia: ', registrados)

final = pd.read_csv(FINAL / 'homeoffice_final.csv')
comprobar('3. Balance y trazabilidad', 'Ningún registro se pierde sin explicación',
          balance_tabla['cuadra'].all() and registrados == contados
          and balance['01_homeoffice']['entrada'] - sum(contados.values()) == len(final),
          f'{miles(balance["01_homeoffice"]["entrada"])} crudos = {miles(len(final))} finales + '
          f'{sum(contados.values())} excluidos, todos con su motivo')

,etapa,entrada,salida,excluidas,motivos,cuadra
0,01_categorias,5080,4412,668,"duplicados 605, invalidas 56, precios_excluido...",True
1,01_homeoffice,3680,2715,965,"ajenas 178, duplicados 787",True
2,02_homeoffice,2715,2714,1,errores 1,True
3,03_homeoffice,2714,2714,0,—,True
4,04_homeoffice,2714,2714,0,—,True


Exclusiones contadas:    {'duplicados': 787, 'ajenas': 178, 'errores': 1}
Registros en evidencia:  {'duplicados': 787, 'ajenas': 178, 'errores': 1}
OK  3. Balance y trazabilidad: 3 680 crudos = 2 714 finales + 966 excluidos, todos con su motivo


## Prueba 4 — El dataset final cumple sus invariantes

Reglas que todo registro del dataset final debe cumplir, documentadas en el diccionario de datos final.

In [5]:
invariantes = {
    'id_oferta único y sin nulos': final['id_oferta'].notna().all() and final['id_oferta'].is_unique,
    'precio > 0': (final['precio'] > 0).all(),
    '0 ≤ descuento_pct < 100': final['descuento_pct'].between(0, 100, inclusive='left').all(),
    'calificacion entre 1 y 5 (o vacía)': final['calificacion'].dropna().between(1, 5).all(),
    'tiene_descuento ⇔ hay precio_anterior': (final['tiene_descuento'] == final['precio_anterior'].notna()).all(),
    'ofrece_cuotas ⇔ hay numero_cuotas': (final['ofrece_cuotas'] == final['numero_cuotas'].notna()).all(),
    'cada oferta tiene una gama válida': final['gama'].isin(['económica', 'media', 'alta']).all(),
    'las 3 gamas existen en los 6 componentes': (final.groupby('componente')['gama'].nunique() == 3).all()
                                                 and final['componente'].nunique() == 6,
    'dimensiones 2714 × 41': final.shape == (2714, 41),
}
display(pd.DataFrame({'invariante': list(invariantes), 'se cumple': list(invariantes.values())}))

comprobar('4. Invariantes del dataset final', 'El dataset final es consistente',
          all(invariantes.values()), f'{len(invariantes)} reglas cumplidas en {miles(final.shape[0])} × {final.shape[1]}')

,invariante,se cumple
0,id_oferta único y sin nulos,True
1,precio > 0,True
2,0 ≤ descuento_pct < 100,True
3,calificacion entre 1 y 5 (o vacía),True
4,tiene_descuento ⇔ hay precio_anterior,True
5,ofrece_cuotas ⇔ hay numero_cuotas,True
6,cada oferta tiene una gama válida,True
7,las 3 gamas existen en los 6 componentes,True
8,dimensiones 2714 × 41,True


OK  4. Invariantes del dataset final: 9 reglas cumplidas en 2 714 × 41


## Prueba 5 — El almacén SQLite responde consultas

El almacenamiento es una base SQLite con cinco tablas. Se comprueba que cada tabla tiene las mismas filas que su CSV,
se consulta el resultado principal del proyecto (costo por perfil) y se intenta insertar una oferta repetida para ver
que la clave primaria la rechaza (sobre una copia en memoria, sin tocar el archivo).

In [6]:
# closing() cierra la conexión al terminar: en Windows, una conexión abierta bloquea el archivo y el pipeline
# no podría regenerarlo en la siguiente ejecución.
with closing(sqlite3.connect(FINAL / 'home_office.sqlite')) as conexion:
    tablas = pd.read_sql("SELECT name AS tabla FROM sqlite_master WHERE type = 'table' ORDER BY name", conexion)
    tablas['filas'] = [conexion.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0] for t in tablas['tabla']]
    display(tablas)

    perfiles = pd.read_sql("""
        SELECT perfil, ROUND(costo_mediano) AS costo_mediano, ROUND(costo_q1) AS rango_desde,
               ROUND(costo_q3) AS rango_hasta
        FROM perfiles ORDER BY costo_mediano""", conexion)
    display(perfiles)

    excluidos = pd.read_sql("""
        SELECT archivo, COUNT(*) AS registros FROM excluidos GROUP BY archivo ORDER BY archivo""", conexion)
    display(excluidos)

    memoria = sqlite3.connect(':memory:')
    conexion.backup(memoria)
try:
    memoria.execute('INSERT INTO publicaciones (id_oferta) SELECT id_oferta FROM publicaciones LIMIT 1')
    rechaza_duplicado = False
except sqlite3.IntegrityError as error:
    rechaza_duplicado = True
    print(f'Inserción de un id_oferta repetido rechazada: {error}')
finally:
    memoria.close()

filas_csv = {'publicaciones': len(final), 'perfiles': manifiesto_despues['perfiles_home_office.csv']['filas'],
             'precios_por_gama': manifiesto_despues['precios_por_gama.csv']['filas'],
             'ofertas_comunes': manifiesto_despues['integracion_ofertas_comunes.csv']['filas']}
coinciden = all(tablas.set_index('tabla').loc[t, 'filas'] == n for t, n in filas_csv.items())
comprobar('5. Almacén SQLite', 'Los datos se almacenan y consultan con SQL',
          len(tablas) == 5 and coinciden and rechaza_duplicado and len(perfiles) == 3,
          f'5 tablas iguales a sus CSV; perfiles de S/ {miles(round(perfiles.costo_mediano.min()))} a '
          f'S/ {miles(round(perfiles.costo_mediano.max()))}; la clave rechaza duplicados')

,tabla,filas
0,excluidos,1079
1,ofertas_comunes,767
2,perfiles,3
3,precios_por_gama,18
4,publicaciones,2714


,perfil,costo_mediano,rango_desde,rango_hasta
0,económico,3644.0,2812.0,4202.0
1,medio,6332.0,5491.0,7373.0
2,alto,12323.0,10464.0,16239.0


,archivo,registros
0,01_ajenas_al_componente.csv,178
1,01_duplicados.csv,787
2,01_inconsistencias.csv,43
3,01_reasignadas.csv,11
4,02_atipicos.csv,51
5,02_correcciones.csv,8
6,02_excluidos_por_error.csv,1


Inserción de un id_oferta repetido rechazada: UNIQUE constraint failed: publicaciones.id_oferta
OK  5. Almacén SQLite: 5 tablas iguales a sus CSV; perfiles de S/ 3 644 a S/ 12 323; la clave rechaza duplicados


## Prueba 6 — El pipeline detecta un dato corrupto y se detiene

Se copia el proyecto a una carpeta temporal, se **duplica a propósito una oferta** en la salida de la etapa 02 y se
ejecuta `python pipeline.py --desde 03`. El resultado esperado es que el pipeline **falle en la etapa 03**, nombre el
invariante violado (`id_oferta` repetido) y salga con un código distinto de 0, sin ejecutar las etapas siguientes.

> **Nota:** la palabra «FALLÓ» que aparece en la salida de esta celda **es el resultado esperado**: la prueba se
> supera precisamente porque el pipeline rechaza el dato corrupto. La línea final `OK  6.` lo confirma.

In [7]:
with tempfile.TemporaryDirectory() as carpeta:
    copia = Path(carpeta)
    shutil.copy2(RAIZ / 'pipeline.py', copia)
    shutil.copytree(RAIZ / 'notebooks', copia / 'notebooks')
    for sub in ('intermedio', 'evidencia', 'final'):
        shutil.copytree(RAIZ / 'data' / sub, copia / 'data' / sub)
    (copia / 'docs' / 'parte_2' / 'figuras').mkdir(parents=True)

    tratado = copia / 'data' / 'intermedio' / '02_homeoffice_tratado.csv'
    datos = pd.read_csv(tratado, dtype=str, keep_default_na=False)
    pd.concat([datos, datos.head(1)]).to_csv(tratado, index=False)
    print(f'Oferta duplicada a propósito: {datos.iloc[0]["id_oferta"]}')

    fallido, _ = ejecutar_pipeline('--desde', '03', carpeta=copia)

print('Salida del pipeline sobre la copia con el dato corrupto (se espera que se detenga en la etapa 03):')
print('   ' + fallido.stdout.strip())
mensaje = next((linea for linea in fallido.stderr.splitlines() if 'FALLÓ' in linea), fallido.stderr[-500:])
print(f'Mensaje esperado del pipeline (el error fue detectado): {mensaje}')
print()

comprobar('6. Detección de errores', 'Un dato corrupto detiene el pipeline con un mensaje claro',
          fallido.returncode != 0 and '[03] FALLÓ' in fallido.stderr and 'id_oferta repetido' in fallido.stderr
          and '[04]' not in fallido.stdout,
          f'se detuvo en la etapa 03 (código {fallido.returncode}) por id_oferta repetido; '
          'las etapas 04–06 no se ejecutaron')

Oferta duplicada a propósito: MPE1082474800


Salida del pipeline sobre la copia con el dato corrupto (se espera que se detenga en la etapa 03):
   [03] Integración y reducción ...
Mensaje esperado del pipeline (el error fue detectado): [03] FALLÓ: AssertionError: Entrada inválida: id_oferta repetido (invariante de unicidad)

OK  6. Detección de errores: se detuvo en la etapa 03 (código 1) por id_oferta repetido; las etapas 04–06 no se ejecutaron


## Resumen

In [8]:
resumen = pd.DataFrame(resultados)
display(resumen)
print(f'{len(resumen)} de 6 pruebas superadas: el pipeline funciona, es reproducible y detecta errores.')

,prueba,qué demuestra,resultado
0,1. Pipeline completo,El pipeline es funcional de punta a punta,"OK · 6 de 6 etapas OK en 51 s, código de salida 0"
1,2. Reproducibilidad,Dos ejecuciones producen exactamente los mismo...,OK · 4 CSV finales con el mismo SHA-256 antes ...
2,3. Balance y trazabilidad,Ningún registro se pierde sin explicación,OK · 3 680 crudos = 2 714 finales + 966 exclui...
3,4. Invariantes del dataset final,El dataset final es consistente,OK · 9 reglas cumplidas en 2 714 × 41
4,5. Almacén SQLite,Los datos se almacenan y consultan con SQL,OK · 5 tablas iguales a sus CSV; perfiles de S...
5,6. Detección de errores,Un dato corrupto detiene el pipeline con un me...,OK · se detuvo en la etapa 03 (código 1) por i...


6 de 6 pruebas superadas: el pipeline funciona, es reproducible y detecta errores.
